# Module 7b: exercises

These use `service_desk.csv` (four differently shaped variables) and `app_trial.csv`. Where randomness appears the generator is seeded.

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

## Exercise 1

**Shape by hand.** Write `skewness(x)` and `excess_kurtosis(x)` using the central-moment formulas
$g_1=m_3/m_2^{3/2}$ and $g_2=m_4/m_2^2-3$ (with $m_k=\frac1n\sum(x_i-\bar x)^k$). Then store the skewness of `response_minutes` in `skew_response` and of `satisfaction_pct` in `skew_satisfaction`.

In [ ]:
import math

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.mixture import GaussianMixture

desk = pd.read_csv(DATA / "service_desk.csv")
trial = pd.read_csv(DATA / "app_trial.csv")

def skewness(x):
    # YOUR CODE HERE
    pass

def excess_kurtosis(x):
    # YOUR CODE HERE
    pass

skew_response = None       # YOUR CODE HERE
skew_satisfaction = None   # YOUR CODE HERE

In [ ]:
probe = np.random.default_rng(0).gamma(2, size=200)
assert abs(skewness(probe) - stats.skew(probe)) < 1e-9
assert abs(excess_kurtosis(probe) - stats.kurtosis(probe)) < 1e-9
assert abs(skew_response - 2.09) < 0.01 and skew_satisfaction < -0.8
print(f"✅ Correct! response times skew {skew_response:.2f} (right tail), satisfaction {skew_satisfaction:.2f} (left tail, capped at 100)")

## Exercise 2

**Maximum likelihood by formula.** Write `mle_exponential_rate(x)` (the rate $\hat\lambda=1/\bar x$), `mle_poisson(x)` ($\hat\lambda=\bar x$) and `mle_lognormal(x)` returning the tuple $(\hat\mu,\hat\sigma)$
from the mean and standard deviation (with `ddof=0`) of $\ln x$. Apply them: `lam_tickets` for `tickets`, and `mu_resp, sigma_resp` for `response_minutes`.

In [ ]:
def mle_exponential_rate(x):
    # YOUR CODE HERE
    pass

def mle_poisson(x):
    # YOUR CODE HERE
    pass

def mle_lognormal(x):
    # YOUR CODE HERE
    pass

lam_tickets = None                 # YOUR CODE HERE
mu_resp = sigma_resp = None        # YOUR CODE HERE

In [ ]:
x = desk["response_minutes"].to_numpy()
assert abs(mle_exponential_rate(x) - 1 / stats.expon.fit(x, floc=0)[1]) < 1e-9
shape, loc, scale = stats.lognorm.fit(x, floc=0)
assert abs(mu_resp - math.log(scale)) < 1e-4 and abs(sigma_resp - shape) < 1e-4
assert abs(lam_tickets - 4.3708) < 1e-3
print(f"✅ Correct! lambda-hat = {lam_tickets:.3f}; lognormal mu = {mu_resp:.3f}, sigma = {sigma_resp:.3f}")

## Exercise 3

**Compare models with AIC.** Write `aic(log_likelihood, k)` $=2k-2\ell$. Using the fitted parameters given, compute the log-likelihood of `response_minutes` under the normal and lognormal models (sum of `logpdf`), their AICs
(`aic_normal`, `aic_lognormal`, each with 2 parameters), and set `best` to the name of the model with the lower AIC (`"normal"` or `"lognormal"`).

In [ ]:
x = desk["response_minutes"].to_numpy()
params_norm = stats.norm.fit(x)
params_logn = stats.lognorm.fit(x, floc=0)

def aic(log_likelihood, k):
    # YOUR CODE HERE
    pass

aic_normal = aic_lognormal = None   # YOUR CODE HERE
best = None                         # YOUR CODE HERE

In [ ]:
assert aic(-100, 3) == 206
assert abs(aic_normal - 3061.1) < 0.5 and abs(aic_lognormal - 2819.7) < 0.5, (aic_normal, aic_lognormal)
assert best == "lognormal"
print(f"✅ Correct! AIC normal {aic_normal:.0f} vs lognormal {aic_lognormal:.0f}: the lognormal is far better.")

## Exercise 4

**Q-Q coordinates.** Write `qq_points(data, dist=stats.norm)` returning `(theoretical, sample)`: the sorted data, and the distribution's quantiles `dist.ppf(p)` at the plotting positions $p_i=(i-0.5)/n$.

In [ ]:
def qq_points(data, dist=stats.norm):
    # YOUR CODE HERE
    pass

In [ ]:
data = [5.0, 1.0, 3.0, 2.0, 4.0]
th, sm = qq_points(data)
assert list(sm) == [1, 2, 3, 4, 5], "The sample quantiles are the sorted data"
assert np.allclose(th, stats.norm.ppf([0.1, 0.3, 0.5, 0.7, 0.9]))
th2, _ = qq_points(data, stats.expon)
assert np.allclose(th2, stats.expon.ppf([0.1, 0.3, 0.5, 0.7, 0.9]))
assert abs(th[2]) < 1e-12 and np.all(np.diff(th) > 0)
print("✅ Correct!")

## Exercise 5

**Kolmogorov–Smirnov by hand.** Write `ks_statistic(data, cdf)` returning $D=\max_i\max\big(\tfrac in-F(x_{(i)}),\,F(x_{(i)})-\tfrac{i-1}n\big)$ for sorted data $x_{(i)}$.
Then store `d_lognormal`, the statistic for `response_minutes` against the fitted lognormal's CDF.

In [ ]:
def ks_statistic(data, cdf):
    # YOUR CODE HERE
    pass

fitted = stats.lognorm(*stats.lognorm.fit(x, floc=0))
d_lognormal = None   # YOUR CODE HERE

In [ ]:
probe = np.random.default_rng(1).normal(0, 1, 60)
assert abs(ks_statistic(probe, stats.norm.cdf) - stats.kstest(probe, stats.norm.cdf).statistic) < 1e-12
assert abs(d_lognormal - 0.03295) < 1e-4, d_lognormal
print(f"✅ Correct! D = {d_lognormal:.4f}: the lognormal CDF is never more than {d_lognormal:.1%} away from the data's CDF")

## Exercise 6

**Is it Poisson?** `observed` holds the number of hours with ≤1, 2, 3, …, 8 and ≥9 tickets. Compute `expected` counts under a Poisson with the fitted rate `lam` (first bin: $P(X\le1)$; last bin: $P(X\ge9)$),
then `chi2` and the p-value `p_value` using **7 degrees of freedom** (9 bins − 1 − 1 estimated parameter).

In [ ]:
tickets = desk["tickets"].to_numpy()
n = len(tickets)
lam = tickets.mean()
observed = np.array([(tickets <= 1).sum()] + [(tickets == k).sum() for k in range(2, 9)] + [(tickets >= 9).sum()])

expected = None   # YOUR CODE HERE (array of 9 expected counts)
chi2 = None       # YOUR CODE HERE
p_value = None    # YOUR CODE HERE

In [ ]:
assert len(expected) == 9 and abs(expected.sum() - n) < 1e-6, "Expected counts must add up to n"
assert abs(chi2 - 4.99) < 0.02, f"chi2 = {chi2}"
assert abs(p_value - 0.66) < 0.02, f"p = {p_value}"
print(f"✅ Correct! chi2 = {chi2:.2f}, p = {p_value:.2f}: no evidence against a Poisson model")

## Exercise 7

**Transform and report as a ratio.** Compute `skew_raw` and `skew_log` for `minutes_active` in `trial`, and write `ratio_ci(a, b)` that takes two arrays, runs a Welch t-test on their **logs**, and returns `(ratio, low, high)`:
the ratio of geometric means `exp(mean(log a) - mean(log b))` and its 95% interval, back-transformed with `exp`. Apply it to the gamified (`a`) vs control (`b`) users' minutes as `ratio, low, high`.

In [ ]:
minutes = trial["minutes_active"]
skew_raw = skew_log = None   # YOUR CODE HERE

def ratio_ci(a, b):
    # YOUR CODE HERE
    pass

gamified = trial.loc[trial["group"] == "gamified", "minutes_active"].to_numpy()
control = trial.loc[trial["group"] == "control", "minutes_active"].to_numpy()
ratio, low, high = None, None, None   # YOUR CODE HERE

In [ ]:
assert skew_raw > 2 and abs(skew_log) < 0.5, (skew_raw, skew_log)
assert abs(ratio - 1.395) < 0.01 and abs(low - 1.124) < 0.01 and abs(high - 1.731) < 0.01, (ratio, low, high)
r2, l2, h2 = ratio_ci([2.0, 4.0, 8.0], [1.0, 2.0, 4.0])
assert abs(r2 - 2.0) < 1e-9 and l2 < r2 < h2
print(f"✅ Correct! Gamified users spent {ratio:.2f}x the minutes (95% CI {low:.2f} to {high:.2f}). Skewness fell from {skew_raw:.1f} to {skew_log:.2f}")

## Exercise 8

**Probabilities from distributions.** Tickets arrive at rate 4.2 per hour (Poisson counts, exponential gaps). Use `scipy.stats` to compute:
`p_at_least_8` (P(8 or more tickets in an hour)), `p_none` (P(no tickets in an hour)), `p_gap_over_30` (P(gap between tickets exceeds 30 minutes)), and `median_gap_min` (the median gap in minutes).
Also for IQ ~ Normal(100, 15): `p_above_130` and `iq_95th` (the 95th percentile).

In [ ]:
rate = 4.2
p_at_least_8 = p_none = p_gap_over_30 = median_gap_min = None   # YOUR CODE HERE
p_above_130 = iq_95th = None                                    # YOUR CODE HERE

In [ ]:
assert abs(p_at_least_8 - 0.0639) < 1e-3, "P(X >= 8) = 1 - P(X <= 7) = sf(7)"
assert abs(p_none - 0.0150) < 1e-3
assert abs(p_gap_over_30 - 0.1225) < 1e-3
assert abs(median_gap_min - 9.9) < 0.05
assert abs(p_above_130 - 0.0228) < 1e-3 and abs(iq_95th - 124.67) < 0.05
print("✅ Correct!")

## Exercise 9

**How many groups?** Fit Gaussian mixtures with 1, 2, 3 and 4 components to `call_minutes` (`GaussianMixture(n_components=k, random_state=0)`), record the BIC of each in the dictionary `bics`
(key = number of components), and store the best (lowest BIC) number as `best_k`. Also store the mixing weights of that best model, sorted ascending, as `weights`.

In [ ]:
calls = desk[["call_minutes"]].to_numpy()
bics = {}
# YOUR CODE HERE
best_k = None
weights = None

In [ ]:
assert set(bics) == {1, 2, 3, 4}
assert best_k == 2, f"best k = {best_k}"
assert np.allclose(weights, [0.39, 0.61], atol=0.03), weights
print(f"✅ Correct! Two groups: about {weights[1]:.0%} short calls and {weights[0]:.0%} long calls")

## Exercise 10

**Which distribution fits the story?** Match each situation to its most natural model by filling the dictionary `answers` with one of `"binomial"`, `"poisson"`, `"exponential"`, `"lognormal"`, `"beta"`, `"normal"`.

* **a)** The number of emails arriving in an hour
* **b)** The time between consecutive emails
* **c)** Household incomes in a city (strongly right-skewed, positive)
* **d)** A website's daily conversion rate, always between 0 and 1
* **e)** The number of heads in 20 coin flips
* **f)** Repeated measurements of one object's length with symmetric random error

In [ ]:
answers = {"a": None, "b": None, "c": None, "d": None, "e": None, "f": None}   # YOUR CODE HERE

In [ ]:
expected = {"a": "poisson", "b": "exponential", "c": "lognormal", "d": "beta", "e": "binomial", "f": "normal"}
for key, want in expected.items():
    assert answers[key] == want, f"({key}) think about the data-generating story"
print("✅ Correct!")